# Lab 05: 卷積與池化效果視覺化 — 使用真實相片

在本單元中，我們不訓練模型，而是透過視覺化來直觀理解 **卷積 (Convolution)** 與 **池化 (Pooling)** 這兩個 CNN 的核心操作。
我們將從網路上下載一張真實的相片，設計不同的「卷積核 (Kernel/Filter)」並套用其上，觀察卷積是如何從真實的複雜相片中提取邊緣與特定特徵的！

### 任務 1: 匯入套件並從網路下載真實影像
我們使用 Python 的 `urllib` 從維基百科下載一張貓咪照片，並使用 `PIL.Image` 將其轉為灰階（單通道），方便我們進行二維卷積運算。
*(貼心設計：若電腦教室因網路限制無法下載，程式會自動繪製幾何圖形作為備用，確保實作不中斷！)*

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
from PIL import Image
import io

# 穩定的維基百科貓咪照片 URL
img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/500px-Cat03.jpg"

try:
    print("正在從網路下載真實貓咪相片...")
    # 下載圖片
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    req = urllib.request.Request(img_url, headers=headers)
    with urllib.request.urlopen(req, timeout=5) as response:
        img_data = response.read()
    
    # 轉為 PIL Image 並轉為灰階
    image = Image.open(io.BytesIO(img_data)).convert('L')
    image = image.resize((256, 256)) # 縮放到 256x256 加快處理速度
    image_np = np.array(image, dtype=np.float32) / 255.0
    print("下載完成，已成功轉換為 256x256 灰階矩陣！")
except Exception as e:
    print(f"網路下載失敗 ({e})。啟動備用方案：自動生成幾何圖形照片...")
    image_np = np.zeros((256, 256), dtype=np.float32)
    image_np[50:150, 50:150] = 1.0
    y, x = np.ogrid[:256, :256]
    circle_mask = (x - 180)**2 + (y - 180)**2 <= 40**2
    image_np[circle_mask] = 1.0

# 繪製原始圖像
plt.figure(figsize=(5, 5))
plt.imshow(image_np, cmap='gray')
plt.title("Original Grayscale Image")
plt.axis('off')
plt.show()

### 任務 2: 將影像轉換為 PyTorch 的 4D 張量
PyTorch 的二維卷積層 `F.conv2d` 期待的輸入形狀為：
`[批次大小(Batch), 通道數(Channel), 圖片高度(Height), 圖片寬度(Width)]`。

請使用 `unsqueeze()` 增加維度，將其形狀轉換為 `[1, 1, 256, 256]`。

In [ ]:
# 將 Numpy 陣列轉為 PyTorch 張量
image_tensor = torch.tensor(image_np, dtype=torch.float32)

# (已幫你完成) 增加 Batch 與 Channel 維度，使其形狀變為 [1, 1, 256, 256]
image_tensor = image_tensor.unsqueeze(0).unsqueeze(0)
print("影像張量的形狀:", image_tensor.shape)

### 任務 3: 定義卷積核 (Kernels)
卷積核就是一個 $3 \times 3$ 的矩陣。不同的數值組合代表不同的濾鏡效果：
1. **垂直邊緣偵測 (Sobel X)**：計算左右兩側的像素差，抓出垂直邊線。
2. **水平邊緣偵測 (Sobel Y)**：計算上下兩側的像素差，抓出水平邊線。
3. **影像模糊 (Blur)**：取周圍像素的平均值，淡化細節。
4. **影像銳利化 (Sharpen)**：加強中心像素，削弱周圍像素。

In [ ]:
# 1. 垂直邊緣偵測核
sobel_x = torch.tensor([[-1., 0., 1.],
                       [-2., 0., 2.],
                       [-1., 0., 1.]])

# 2. 水平邊緣偵測核
sobel_y = torch.tensor([[-1., -2., -1.],
                       [ 0.,  0.,  0.],
                       [ 1.,  2.,  1.]])

# 3. TODO: 請定義一個 3x3 的平均模糊核 (每個位置的值都是 1/9)
# 提示：blur_kernel = torch.ones(3, 3) / 9
# ??? (請在此填寫你的答案)

# 4. 銳利化核
sharpen_kernel = torch.tensor([[ 0., -1.,  0.],
                               [-1.,  5., -1.],
                               [ 0., -1.,  0.]])

# 將這些卷積核形狀重塑為 [輸出通道, 輸入通道, 高, 寬] = [1, 1, 3, 3]
kernels = {
    "Sobel X (Vertical Edge)": sobel_x.view(1, 1, 3, 3),
    "Sobel Y (Horizontal Edge)": sobel_y.view(1, 1, 3, 3),
    "Blur Filter": blur_kernel.view(1, 1, 3, 3),
    "Sharpen Filter": sharpen_kernel.view(1, 1, 3, 3)
}

### 任務 4: 執行卷積並觀察效果
我們使用 `F.conv2d` 將這些濾鏡套用到我們的圖片上，並畫出來對比。

In [ ]:
plt.figure(figsize=(15, 8))

# 先畫原始圖
plt.subplot(2, 3, 1)
plt.imshow(image_np, cmap='gray')
plt.title("Original Image")
plt.axis('off')

# 逐一計算並畫出濾鏡效果
plot_idx = 2
for name, kernel_tensor in kernels.items():
    # padding=1 表示邊緣補零，這樣輸出影像尺寸才會與輸入相同 (256x256)
    conv_out = F.conv2d(image_tensor, kernel_tensor, padding=1)
    
    # 移除批次與通道維度以利繪圖
    out_img = conv_out.squeeze().numpy()
    
    plt.subplot(2, 3, plot_idx)
    plt.imshow(out_img, cmap='gray')
    plt.title(name)
    plt.axis('off')
    plot_idx += 1

plt.tight_layout()
plt.show()

### 任務 5: 池化 (Pooling) 降維效果
最大池化 `MaxPool2d` 會在指定的區域（例如 $2 \times 2$）內挑選出數值最大的像素點。這會減少影像尺寸並保留強烈特徵。

In [ ]:
# TODO: 呼叫 F.max_pool2d，設定 kernel_size=2, stride=2
# 這代表會將圖片寬高各減半
pooled_tensor = F.max_pool2d(image_tensor, kernel_size=2, stride=2)
pooled_img = pooled_tensor.squeeze().numpy()

print(f"原始圖片大小: {image_tensor.shape[2:]}")
print(f"最大池化後圖片大小: {pooled_img.shape}")

# 繪圖對比
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(image_np, cmap='gray')
plt.title(f"Original {image_np.shape}")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(pooled_img, cmap='gray')
plt.title(f"Pooled {pooled_img.shape}")
plt.axis('off')
plt.show()

---

## Lab 05 學習重點小結
### 核心概念掌握
- 卷積核 (Kernel) 是一個小型可學習的權重矩陣，在影像上滑動做元素乘積加總。
- Sobel 運算子是一種邊緣偵測卷積核，可提取水平或垂直邊緣特徵。
- 最大池化 (Max Pooling) 取區域最大值，降低特徵圖尺寸同時保留最顯著特徵。
- 卷積層的平移不變性使 CNN 能辨識不同位置的相同特徵。

### 快速自評測驗

**Q1. 3×3 卷積核在 32×32 的影像上滑動 (stride=1, no padding)，輸出特徵圖大小為？**
- A. 32×32
- B. 30×30
- C. 29×29

<details><summary>查看解答</summary>

**答案：B — 輸出尺寸 = (32 - 3 + 1) = 30，故輸出為 30×30。**

</details>

**Q2. 最大池化 (Max Pooling) 的主要作用為？**
- A. 增加影像解析度
- B. 降維並保留最顯著特徵
- C. 增加模型參數量

<details><summary>查看解答</summary>

**答案：B — 池化層降低特徵圖大小，減少計算量，同時增強位移不變性。**

</details>

### 完成確認清單
在繼續下一個 Lab 前，請確認你能做到：
- [ ] 我可以用一句話解釋「卷積核 (Kernel) 是一個小型可學習的權重矩陣，在影像」
- [ ] 我可以用一句話解釋「Sobel 運算子是一種邊緣偵測卷積核，可提取水平或垂直邊緣特」
- [ ] 我可以用一句話解釋「最大池化 (Max Pooling) 取區域最大值，降低特徵」
- [ ] 我的程式碼成功執行並得到預期輸出
